# Module 1 — Build a local text-to-SQL agent

In Module 0 you stood up the data layer (an Athena database over student data in S3). Now you'll
build an agent that answers **natural-language questions** about that data — it writes the SQL,
runs it on Athena, and analyzes the results.

The big idea of this module: **context engineering**. The same agent, given the same Athena tool,
behaves completely differently depending on whether it has *domain context* (a CLAUDE.md workflow +
skills + table metadata). We'll see it both ways:

- **Part 1A — no skills:** the agent has the query tool but no guidance. It guesses column names.
- **Part 1B — with skills:** the agent loads a skill, reads the table metadata, then writes correct SQL.


## Setup

Run the cell below to install all dependencies and register the Jupyter kernel.
After it completes, **select the `agentic-analytics-module-1-local-agent` kernel** from the kernel picker (top-right)
and continue with the rest of the notebook.

### Setup step 1

When you run the first script, it will ask you to select a environment

![](images/select-system-python.png)

and you can select the global env for now, and in this case it is 3.11.15 but this version may change

![](images/select-python-global-env.png)

once selected, you can rerun the setup.sh script

In [1]:
!bash setup.sh

Using CPython 3.11.14
Creating virtual environment at: .venv
Resolved 99 packages in 207ms                                        
⠙ Preparing packages... (0/12)                                                  
⠙ Preparing packages... (0/12)------------------     0 B/45.06 KiB           
⠙ Preparing packages... (0/12)------------------ 14.84 KiB/45.06 KiB         
⠙ Preparing packages... (0/12)------------------ 14.84 KiB/45.06 KiB         
sqlparse             ------------------------------ 14.84 KiB/45.06 KiB
⠙ Preparing packages... (0/12)------------------     0 B/119.90 KiB          
sqlparse             ------------------------------ 14.84 KiB/45.06 KiB
⠙ Preparing packages... (0/12)------------------ 14.84 KiB/119.90 KiB        
sqlparse             ------------------------------ 14.84 KiB/45.06 KiB
⠙ Preparing packages... (0/12)------------------ 30.84 KiB/119.90 KiB        
sqlparse             ------------------------------ 14.84 KiB/45.06 KiB
⠙ Preparing packages... (0/12)--

### Setup step 2

Once you see the dependencies and kernel spec `agentic-analytics-module-1-local-agent` are installed per the message from the last step, please refresh your browser (not refresh kernel but browser)

![](images/refresh-browser.png)

and once refreshed, click on the button (it probably shows a python version 3.11.15) you used to select kernel in the preview section

![](images/current-python.png)

it will show you the option to select another kernel and please click

![](images/select-another-kernel.png)

once clicked, you will see the option to select a Jupyter kernel — please click on "Jupyter Kernel"

![](images/select-jupyter-kernel.png)

once clicked, you can see our registered module kernel. The screenshot below shows module-1 as an example, but ***please select `agentic-analytics-module-1-local-agent` since you are working on Module 1***

![](images/example-select-module-1-jupter-kernel.png)

once selected, you will see it as your active kernel. Again the screenshot shows module-1 as an example, ***please select accordingly depending on which module you are working on — for this module, select `agentic-analytics-module-1-local-agent`***

![](images/example-module-1-jupyter-kernel-selected.png)

Confirm the environment: Bedrock model access + the Athena data layer from Module 0.

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

import boto3
acct = boto3.client("sts").get_caller_identity()["Account"]
region = os.getenv("AWS_REGION", "us-west-2")

# Point the agent at the bucket Module 0 created (derived from your account id).
os.environ.setdefault("ATHENA_OUTPUT_LOCATION", f"s3://student-analytics-agent-{acct}/athena-results/")
os.environ.setdefault("ATHENA_DATABASE", "student_analytics")
print("Region:", region)
print("Athena DB:", os.environ["ATHENA_DATABASE"])
print("Athena output:", os.environ["ATHENA_OUTPUT_LOCATION"])

Region: us-east-1
Athena DB: student_analytics
Athena output: s3://student-analytics-agent-490753044219/athena-results/


---
## The one tool the agent needs: `execute_athena_query`

Everything hangs off a single tool that runs a SQL SELECT on Athena and downloads the results.
It's defined once in `analytics_agent/agent.py` and exposed to the agent as an in-process MCP server.
The SQL validator (`tools/sql_validator.py`) rejects anything that isn't a SELECT — a guardrail so the
agent can't mutate data.

## Part 1A — the agent *without* skills

Here we wire up the Athena tool but run the agent in **isolation mode** (`setting_sources=[]`): no
CLAUDE.md, no skills, no table metadata. We ask a real question and watch it *guess* at the schema.

In [9]:
import sys, uuid, csv
from pathlib import Path
sys.path.insert(0, "analytics_agent")
import agent as analytics
from claude_agent_sdk import ClaudeSDKClient, ClaudeAgentOptions

async def run(options, prompt, request_id=None):
    """Run the agent, print its text output, then show a summary of SQL + results + answer."""
    text = []
    async with ClaudeSDKClient(options=options) as client:
        await client.query(prompt)
        async for msg in client.receive_response():
            for block in getattr(msg, "content", []) or []:
                t = getattr(block, "text", None)
                if t:
                    print(t, end="")
                    text.append(t)

    # End-of-run summary: SQL queries + their results + final answer
    if request_id:
        results_dir = Path(f"results/raw/{request_id}")
        sql_files = sorted(results_dir.glob("*.sql")) if results_dir.exists() else []
        if sql_files:
            print("\n\n" + "=" * 60)
            print("  SUMMARY: SQL QUERIES + RESULTS")
            print("=" * 60)
            for sf in sql_files:
                csv_file = sf.with_suffix(".csv")
                print(f"\n  >> {sf.stem}")
                print(f"     SQL: {sf.read_text().strip()}")
                if csv_file.exists():
                    with open(csv_file) as f:
                        rows = list(csv.reader(f))
                    if len(rows) <= 6:
                        for row in rows:
                            print(f"     {'  |  '.join(row)}")
                    else:
                        for row in rows[:4]:
                            print(f"     {'  |  '.join(row)}")
                        print(f"     ... ({len(rows)-1} rows total)")
            print("\n" + "-" * 60)
            print("  FINAL ANSWER:")
            print("-" * 60)
            # Extract just the last substantive paragraph as the answer
            answer = "".join(text).strip().split("\n")
            for line in answer[-5:]:
                if line.strip():
                    print(f"  {line.strip()}")
            print("=" * 60)

    return "".join(text)

# Build options that REUSE the Athena MCP server but with NO project context.
rid = "m1a-" + uuid.uuid4().hex[:8]
base = analytics.build_agent_options(request_id=rid)
no_skills = ClaudeAgentOptions(
    system_prompt="You answer questions about student data using the execute_athena_query tool.",
    allowed_tools=["Bash", "Read", "Write", "mcp__athena__execute_athena_query"],
    mcp_servers=base.mcp_servers,     # same Athena tool
    setting_sources=[],               # ← isolation: no CLAUDE.md, no skills, no metadata
    cwd=analytics.AGENT_DIR,
    max_turns=15,
)

_ = await run(no_skills, "How many students have an outstanding balance? show me the final count.", request_id=rid)

-------------------- SQL QUERY --------------------
SELECT COUNT(*) AS students_with_outstanding_balance FROM students WHERE balance > 0
---------------------------------------------------
Query submitted with ID: 6a6b10f1-3393-458f-87eb-f3a4501eaf96
Waiting for query to complete...
-------------------- SQL QUERY --------------------
SHOW TABLES
---------------------------------------------------
-------------------- SQL QUERY --------------------
SELECT table_name FROM information_schema.tables WHERE table_schema = 'student_analytics'
---------------------------------------------------
Query submitted with ID: a81d7b02-38ff-49a3-8e8e-c510adb7864d
Waiting for query to complete...
Query completed successfully!
  - Data scanned: 0.00 MB
  - Execution time: 0.47 seconds
Results downloaded to: results/raw/m1a-9b0cec0a/tables_list_2026_06_07_10_40_50.csv
Now I can see the available tables. Let me check the `financial_summary_by_student` table for outstanding balances.-------------------- SQ

Notice what happened: with no metadata, the agent has to *guess* table and column names
(`outstanding_balance`? `balance`? which table?). It may run a wrong query, hit an error, or
hedge. It has the tool but not the **knowledge** to use it well.

## Part 1B — the agent *with* skills + context

Now the same question, but through `build_agent_options()` — which sets `setting_sources=["project"]`.
That single change makes the SDK load, from the `analytics_agent/` bundle:

- **CLAUDE.md** — the always-on workflow (load a skill → read metadata → write SQL → run → analyze),
- **`.claude/skills/`** — `enrollment` and `financial` domain skills,
- the **table metadata** the agent reads before writing SQL.

We'll first **inspect** the options object to see exactly what's configured, then run the agent.
Compare `setting_sources` and `allowed_tools` against Part 1A above — that's the difference.

In [10]:
rid = "m1b-" + uuid.uuid4().hex[:8]
with_skills = analytics.build_agent_options(request_id=rid)

# --- Inspect: what does build_agent_options() actually produce? ---
print("=== ClaudeAgentOptions returned by build_agent_options() ===\n")
print(f"  system_prompt:   {with_skills.system_prompt[:120]}...")
print(f"  allowed_tools:   {with_skills.allowed_tools}")
print(f"  mcp_servers:     {list(with_skills.mcp_servers.keys())}")
print(f"  setting_sources: {with_skills.setting_sources}")
print(f"  cwd:             {with_skills.cwd}")
print(f"  max_turns:       {with_skills.max_turns}")
print()
print("With setting_sources=['project'], the SDK reads from cwd:")
print("  • CLAUDE.md        → always-on memory")
print("  • .claude/skills/  → domain skills (enrollment, financial)")
print("  • data/metadata/   → table schemas the skills reference")
print("=" * 60)

=== ClaudeAgentOptions returned by build_agent_options() ===

  system_prompt:   You are a Student Analytics AI Agent. You answer natural-language
questions about a university's student data by writing...
  allowed_tools:   ['Skill', 'Read', 'Write', 'Bash', 'mcp__athena__execute_athena_query']
  mcp_servers:     ['athena']
  setting_sources: ['project']
  cwd:             /workshop/sample-agentic-ai-with-claude-agent-sdk-and-amazon-bedrock-agentcore/advanced/agentic-analytics/module-1-local-agent/analytics_agent
  max_turns:       30

With setting_sources=['project'], the SDK reads from cwd:
  • CLAUDE.md        → always-on memory
  • .claude/skills/  → domain skills (enrollment, financial)
  • data/metadata/   → table schemas the skills reference


In [11]:
_ = await run(with_skills, "How many students have an outstanding balance? show me the final count.", request_id=rid)


Base directory for this skill: /workshop/sample-agentic-ai-with-claude-agent-sdk-and-amazon-bedrock-agentcore/advanced/agentic-analytics/module-1-local-agent/analytics_agent/.claude/skills/financial

# Quick Start Guide

**Before writing ANY SQL query:**

1. **Read metadata files** (MANDATORY):
   - `data/metadata/financial_summary_by_student.yaml`
   - `data/metadata/financial_summary_by_student_sample_data.csv`

2. **Identify query pattern** from user's question:
   - "How many..." → Pattern 1 (Simple Counting)
   - "...for each [dimension]" → Pattern 2 (Aggregation by Dimension)
   - "Which majors..." → Pattern 3 (Ranking Dimensions)
   - "Top N students..." → Pattern 4 (Top N)
   - "Breakdown by..." → Pattern 5 (Distribution)

3. **Apply Default Assumptions** - Filter for active students unless explicitly stated otherwise

4. **Check Standard Metric Definitions** for ambiguous terms like "financial support"

5. **Apply pattern guidelines** (see Query Pattern Guidelines section)

6.

Same agent, same tool, same question — but with the domain context it follows the workflow:
loads the right skill, reads `financial_summary_by_student`'s metadata, and queries the real
`has_outstanding_balance` / `outstanding_balance` columns. **That's context engineering.**

---
## The packaged entrypoint: `run_query()`

In real use you don't hand-build options — you call `run_query()` from `agent.py`. It wraps
`build_agent_options()` (the single source of truth) with the streaming loop. **Module 2 deploys
this exact same `build_agent_options()` to AgentCore** — no agent logic gets rewritten for the cloud.

In [ ]:
result, messages = await analytics.run_query(
    "What are the top 5 majors by number of enrolled students?",
    request_id="m1-" + uuid.uuid4().hex[:8],
)
print("\n\n=== final answer ===\n", result)

-------------------- SQL QUERY --------------------
SELECT student_major,
       COUNT(DISTINCT student_id) AS enrolled_students
FROM student_enrollment_analytics
WHERE student_enrollment_status = 'Enrolled'
  AND student_status = 'Active'
  AND course_semester = 'Fall 2025'
GROUP BY student_major
ORDER BY enrolled_students DESC
LIMIT 5
---------------------------------------------------
Query submitted with ID: ee8b73a8-bc57-4ed6-9f12-12bb4a7d17bc
Waiting for query to complete...
Query completed successfully!
  - Data scanned: 13.42 MB
  - Execution time: 0.89 seconds
Results downloaded to: results/raw/m1-b25cf3e1/top_5_majors_by_enrollment_2026_06_07_10_43_29.csv


=== final answer ===
 Here are the **top 5 majors by number of currently enrolled students** (Fall 2025, active students with enrolled status):

| Rank | Major | Enrolled Students |
|------|-------|:-----------------:|
| 1 | **Business** | 394 |
| 2 | **Engineering** | 379 |
| 3 | **English** | 369 |
| 4 | **Physics** | 36

## Recap

- One tool (`execute_athena_query`) + **domain context** (CLAUDE.md + skills + metadata) = a capable
  text-to-SQL analyst.
- `setting_sources=["project"]` is the switch that loads that context from the bundle.
- `build_agent_options()` is the **single source of truth** — the local `run_query()` and the
  deployed entrypoint (Module 2) both call it.

**Next:** Module 2 deploys this agent to AgentCore Runtime and traces it in CloudWatch.